In [1]:
using Pkg
Pkg.activate(".")

  Activating project at `~/code/compositional`


In [2]:
using DataFrames, JLD2, StatsBase, Distributions, Random
using CSV, CodecZlib

In [76]:
df = CSV.read("./Data/books_raw/gutenberg_data.csv", DataFrame)
book = df[6,:];

In [77]:
length(book.Text)

580499

In [59]:
STOPWORDS = Set([
    "me","my","myself","we","our","ours","ourselves","you","your","yours","yourself","yourselves","he","him","his","himself",
    "she","her","hers","herself","it","its","itself","hey","them","their","theirs","themselves","what","which","who","whom",
    "this","that","these","those","am","is","are","was","were","be","been","being","have","has","had","having","do","does","did",
    "doing","an","the","and","but","if","or","because","as","until","while","of","at","by","for","with","about","against","between",
    "into","through","during","before","after","above","below","to","from","up","down","in","out","on","off","over","under","again",
    "further","then","once","here","there","when","where","why","how","all","any","both","each","few","more","most","other","some",
    "such","no","nor","not","only","own","same","so","than","too","very","can","will","just","don","should","now",
    "a","b","c","d","e","f","g","h","i","j","k","l","m","n","o","p","q","r","s","t","u","v", "w","x","y","z"
])

function word_counts(text; remove=true)
    # Normalize & split into all words (including stop-words)
    text = lowercase(text)
    # Split on anything that isn't A–Z or a–z (one or more in a row)
    raw_words = filter(!isempty, split(text, r"[^A-Za-z]+"))
    
    # Count everything
    cnt_map = countmap(raw_words)
    nreads = sum(values(cnt_map))
    
    if remove # Remove stop-words from the count map
        filter!(p -> !(p.first in STOPWORDS), cnt_map)
    end
    
    return cnt_map, nreads
end

# prepare empty DataFrame with the right column types
data = DataFrame(
    env        = String[],
    species_id = String[],
    sample_id  = Int[],
    count      = Int[],
    nreads     = Int[],
)

env, sid = book.Bookshelf, book.ID
cnt_map, nreads = word_counts(book.Text, remove=false)
for (word, freq) in cnt_map
    push!(data, (env, word, sid, freq, nreads))
end

unique!(data, [:env, :species_id, :sample_id]);

In [ ]:


# Assuming your DataFrame `df` has columns "species_id" and "freq"
# Let's define the number of words in the new book
N = 1000  # example value

probabilities = data.count ./ data.nreads  # normalized frequencies

# Sample N words based on the probabilities
sampled_indices = rand(Categorical(probabilities), N)

# Build the new book (new DataFrame with the sampled words)
new_book = DataFrame(species_id = df.species_id[sampled_indices])

# Count the occurrences of each species_id in the new book
word_counts = countmap(new_book.species_id)

# Add the counts as a new column in the DataFrame
new_book_with_counts = DataFrame(species_id = unique(new_book.species_id))

# Add the word count column to this new DataFrame
new_book_with_counts.count = [get(word_counts, word, 0) for word in new_book_with_counts.species_id]

# Show the new DataFrame with word counts
println(new_book_with_counts)


In [65]:
S = length(data.species_id) # n species
T = 1000 # n samples
N = 20_000 # n reads

Random.seed!(1234)

# invent unique IDs
species_ids = ["species_$(i)" for i in 1:S]
sample_ids  = ["s$j"  for j in 1:T]

# N_vec = Int.(floor.(rand(Gamma(1.5, N), T)))
N_vec = [N for _ in 1:T]

p = data.count ./ data.nreads

samples = [rand(Multinomial(N_vec[t], p)) for t in 1:T]
count_mat = hcat(samples...)

df_wide = DataFrame(count_mat, sample_ids)
df_wide.species_id .= species_ids

NULL_MODEL = stack(df_wide, Not(:species_id); variable_name = :sample_id, value_name = :count)

# add the nreads column by lookup
idx = parse.(Int, replace.(NULL_MODEL.sample_id, r"^s" => ""))
NULL_MODEL.nreads = N_vec[idx]
NULL_MODEL.env .= "NULL"

# reorder columns if you like
select!(NULL_MODEL, [:env, :species_id, :sample_id, :count, :nreads])

# remove stop words
filter(row -> !(row.species_id in STOPWORDS), NULL_MODEL)

# remove null counts
NULL_MODEL = filter(row -> row.count != 0, NULL_MODEL);

@save "./Data/processed/null_model.jld2" NULL_MODEL

In [64]:
NULL_MODEL

Row,env,species_id,sample_id,count,nreads
,String,String,String,Int64,Int64
1,NULL,species_2,s1,1,16393
2,NULL,species_4,s1,1,16393
3,NULL,species_8,s1,3,16393
4,NULL,species_10,s1,2,16393
5,NULL,species_13,s1,2,16393
6,NULL,species_16,s1,1,16393
7,NULL,species_17,s1,1,16393
8,NULL,species_22,s1,1,16393
9,NULL,species_23,s1,1,16393
